# Đo dung lượng trước/sau khi lọc ảnh trên Google Colab (CPU)

Notebook này tải lại các Version của một **mẫu ngẫu nhiên Publication hoàn chỉnh** qua chính
Stage Source fetch của Engine, rồi đo ba số cho từng Version:

| Số | Ý nghĩa |
|---|---|
| `downloaded_bytes` | byte tải qua mạng (thường là gzip) |
| `content_bytes` | nội dung archive sau giải nén, trước khi chọn `.tex/.bib` |
| `kept_bytes` | `.tex/.bib` giữ lại |

Đồng thời so file giữ lại với bản trong corpus (tên và sha256). Lượt chạy được bọc trong
`src.benchmark`, nên có luôn thời gian, RAM và dung lượng đĩa trên Colab.

**Chuẩn bị (một lần):**
1. Trên máy có corpus, tạo file kế hoạch (chỉ chứa ID, số Version và checksum, không có nội dung):
   `python -m scripts.measure_sizes --corpus data/crawl-smoke --sample 300 --plan-out stats/size-plan.json`
2. Tải `size-plan.json` lên Google Drive, vào thư mục `DRIVE_DIR` ở ô tham số.
3. Thêm `CONTACT_EMAIL` vào Colab Secrets (biểu tượng chìa khóa) và cho notebook truy cập.

**Không chạy song song** với một lượt tải arXiv khác của nhóm: arXiv yêu cầu một kết nối một lúc,
cách nhau tối thiểu 3 giây.

In [ ]:
# ==== Tham số ====
REPO_URL = "https://github.com/hcmus-arxiv-reference/arxiv-reference-matching.git"
REF = "main"                       # nhánh, tag hoặc SHA; commit thực tế được ghi vào run.json
DRIVE_DIR = "/content/drive/MyDrive/arxiv-m1-size-sample"
PLAN_NAME = "size-plan.json"       # file kế hoạch đã tải lên DRIVE_DIR
RUN_ID = None                      # None: tự đặt theo giờ UTC

REPO_DIR = "/content/arxiv-reference-matching"
WORK_ROOT = "/content/size-sample"  # đĩa cục bộ của Colab: số đo đĩa không lẫn độ trễ của Drive

In [ ]:
import hashlib, json, os, platform, shutil, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
RUN_ID = RUN_ID or datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")
RUN_DIR = Path(WORK_ROOT, RUN_ID)
RUN_DIR.mkdir(parents=True, exist_ok=False)


def run(cmd, log, *, check=True, env=None):
    """Chạy lệnh, ghi stdout/stderr ra file; check=True thì mã thoát khác 0 sẽ ném lỗi kèm 20 dòng cuối."""
    with open(log, "w", encoding="utf-8") as stream:
        code = subprocess.run([str(c) for c in cmd], stdout=stream, stderr=subprocess.STDOUT, env=env).returncode
    if check and code:
        tail = Path(log).read_text(encoding="utf-8").splitlines()[-20:]
        raise RuntimeError(f"{cmd[0]} thoát mã {code}:\n" + "\n".join(tail))
    return code

In [ ]:
# Clone repo tại REF và ghi commit thực tế.
if not os.path.isdir(REPO_DIR):
    run(["git", "clone", REPO_URL, REPO_DIR], RUN_DIR / "git-clone.log")
run(["git", "-C", REPO_DIR, "fetch", "--tags", "origin", REF], RUN_DIR / "git-fetch.log")
run(["git", "-C", REPO_DIR, "checkout", "--detach", "FETCH_HEAD"], RUN_DIR / "git-checkout.log")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], RUN_DIR / "pip.log")
print("ref:", REF, "commit:", COMMIT)

In [ ]:
# Bí mật lấy từ Colab Secrets; không in giá trị.
if IN_COLAB:
    from google.colab import userdata
    try:
        os.environ["CONTACT_EMAIL"] = userdata.get("CONTACT_EMAIL")
    except Exception:
        pass
assert os.environ.get("CONTACT_EMAIL"), "Thêm CONTACT_EMAIL trong Colab Secrets (cần cho User-Agent arXiv)"
print("CONTACT_EMAIL: đã đặt")

In [ ]:
# Lấy file kế hoạch từ Drive.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
PLAN = RUN_DIR / PLAN_NAME
shutil.copy(Path(DRIVE_DIR, PLAN_NAME), PLAN)
plan = json.loads(PLAN.read_text())
print("seed:", plan["seed"], "| Publication:", len(plan["publications"]),
      "| Version:", sum(p["versions"] for p in plan["publications"]))

In [ ]:
# Đo: measure_sizes chạy bên trong src.benchmark (thời gian, RAM cả cây tiến trình, đĩa của WORK).
import psutil

WORK = RUN_DIR / "work"
OUT = RUN_DIR / "out"
cmd = [sys.executable, "-m", "src.benchmark", "--data-dir", WORK, "--out", OUT / "bench", "--",
       sys.executable, "-m", "scripts.measure_sizes", "--plan", PLAN, "--work", WORK, "--out", OUT / "sizes"]
machine = {"platform": platform.platform(), "python": sys.version.split()[0], "cpus": os.cpu_count(),
           "ram_gib": round(psutil.virtual_memory().total / 2**30, 1), "colab": IN_COLAB}
started = datetime.now(timezone.utc).isoformat()
code = run(cmd, RUN_DIR / "measure.log", check=False)
RUN = {"run_id": RUN_ID, "ref": REF, "commit": COMMIT, "machine": machine, "started_at": started,
       "finished_at": datetime.now(timezone.utc).isoformat(), "returncode": code,
       "plan_sha256": hashlib.sha256(PLAN.read_bytes()).hexdigest()}
(RUN_DIR / "run.json").write_text(json.dumps(RUN, indent=2))
print("mã thoát:", code)
print("\n".join((RUN_DIR / "measure.log").read_text(encoding="utf-8").splitlines()[-3:]))

In [ ]:
# Kết quả.
summary = json.loads((OUT / "sizes" / "summary.json").read_text())
bench = json.loads((OUT / "bench" / "benchmark.json").read_text())
print(json.dumps(summary, indent=2, ensure_ascii=False))
print(json.dumps({k: v for k, v in bench.items() if not isinstance(v, (list, dict))}, indent=2))

In [ ]:
# Lưu kết quả lên Drive (chỉ số liệu và log; không lưu file .tex đã tải lại).
target = Path(DRIVE_DIR, "runs", RUN_ID)
target.mkdir(parents=True, exist_ok=True)
for item in ["run.json", "measure.log", PLAN_NAME]:
    shutil.copy(RUN_DIR / item, target / item)
shutil.copytree(OUT, target / "out", dirs_exist_ok=True)
print("đã lưu:", target)